# The T3010 panel, and what "failure" means

What this notebook establishes:

1. what the CRA extract actually contains, and where its schema drifts;
2. how 420,000 filings become a clean charity-year panel;
3. why a blank cell means two different things depending on who filed it;
4. two outcome definitions, their base rates, and the censoring that constrains them.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from charity_risk import config, fields, plots
from charity_risk.ingest import read_financials, read_ident
from charity_risk.panel import load_panel, presence_matrix
from charity_risk.outcomes import add_outcomes, outcome_summary, labelled_years

plots.use_project_style()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

print("raw extract :", config.RAW_DIR)
print("years       :", config.YEARS)
print("train / test:", config.TRAIN_YEAR, "/", config.TEST_YEAR)

## 1. What is in the extract

One directory per fiscal-period-end year, each holding the CRA's public T3010
tables. Two of them carry everything this project needs:

* `Ident.csv` — one row per charity: legal name, designation, category, province,
  registration date.
* `Financial Section D & Schedule 6.csv` — one row per return: the balance
  sheet, revenue by source, and expenditure by function.

We deliberately use **only these two**. The 2023 folder is missing
`Financial Section A, B and C.csv`, `Schedule 1` and `Schedule 5`, and building
on files that exist for four years out of five would mean either dropping 2023 —
which is the lookahead year the exit label depends on — or carrying a structural
break through the panel.

In [ ]:
rows = []
for year in config.YEARS:
    files = sorted(p.name for p in (config.RAW_DIR / str(year)).glob("*.csv"))
    rows.append({"year": year, "n_files": len(files),
                 "has_section_abc": any("Section A" in f for f in files),
                 "has_schedule_1": any("Schedule 1" in f for f in files)})
pd.DataFrame(rows).set_index("year")

The schema also drifts between years, and quietly. The 2023 files are UTF-8 with
a byte-order mark while 2019-2022 are Latin-1; the 2023 `Ident.csv` renames three
columns and drops four, including the registration date; and the 2023 financial
file drops a column. `charity_risk.ingest` absorbs all of it, and back-fills
registration date across years within a BN because it is time-invariant.

In [ ]:
ident_2019 = read_ident(2019).columns.tolist()
ident_2023 = read_ident(2023).columns.tolist()
print("Ident columns that are all-null in 2023 (recovered by carry-forward):")
raw_2023 = read_ident(2023)
print([c for c in ident_2019 if c in raw_2023 and raw_2023[c].isna().all()])

## 2. Two filing tiers, one file

The `Financial Indicator` column says whether a return was filed on the short
**Section D** form or the detailed **Schedule 6**. Roughly a third are Section D.

This is not a minor detail. Section D does not ask for cash, receivables,
payables, investment income, compensation or fundraising expense at all. If a
blank is read as zero — the natural default, and the correct one for a Schedule 6
filer who genuinely had no investment income — then every short-form filer is
assigned a cash balance of zero, a compensation ratio of zero, and a fundraising
expense of zero. The resulting "ratios" would encode nothing but the filing tier.

In [ ]:
financial_2021 = read_financials(2021)
tier = financial_2021["filer_tier"]
print(tier.value_counts(dropna=False).to_string(), "\n")

check = ["4200", "4350", "4700", "5100", "4100", "4300", "4580", "4880", "5020"]
coverage = pd.DataFrame({
    "line": check,
    "label": [fields.FIELD_LABELS[c] for c in check],
    "pct_reported_section_D": [100 * financial_2021.loc[tier == "D", c].notna().mean() for c in check],
    "pct_reported_schedule_6": [100 * financial_2021.loc[tier == "6", c].notna().mean() for c in check],
    "collected_on_section_D": [fields.is_collected(c, "D") for c in check],
}).set_index("line")
coverage.round(1)

Every line that `fields.is_collected` marks as Schedule-6-only is reported by
exactly 0.0% of Section D filers — the metadata and the data agree. Downstream,
`panel.apply_tier_missingness` zero-fills blanks only where the filer's tier was
asked the question, and leaves the rest missing. Stated as a rule:

* **asked and left blank = nil.** A Schedule 6 filer that reports total revenue
  but leaves line 4540 empty received no federal government revenue. It is a
  reported zero.
* **never asked = missing.** A Section D filer has no line 4540 on its form.
  Whatever its federal government revenue was, it is bundled into the totals and
  cannot be recovered.

The share of asked lines reported as nil is retained as the feature
`share_nil_lines`. Because a blank *is* a zero, that feature measures how few
distinct kinds of revenue and expenditure an organisation has — not how
carelessly it files.

The same logic drives the harmonised revenue buckets. Section D collects a single
government line (4570); Schedule 6 splits it three ways (4540/4550/4560). Summing
naively would double-count for one tier and undercount for the other.

In [ ]:
for bucket, by_tier in fields.REVENUE_SOURCES_CORE.items():
    print(f"{bucket:32s}  Section D: {' + '.join(by_tier['D']):22s}  Schedule 6: {' + '.join(by_tier['6'])}")

## 3. From filings to a panel

Three cleaning steps, in `charity_risk.panel`:

**De-duplication.** About 0.2% of BNs file two returns with a fiscal period
ending in the same calendar year, nearly always because they changed their
year-end and filed a stub period. We keep the return with the latest period end,
so that stock variables line up with the year label, and record
`n_returns_in_year` so the stub can be identified. That flag turns out to matter
— see the importance ranking in `03-model-results`.

**Tier-aware blanks**, as above.

**Presence bookkeeping** — did this BN file in each year of the window.

In [ ]:
panel = load_panel(columns=["filer_tier", "total_assets", "total_liabilities",
                            "net_assets", "total_revenue", "total_expenditures",
                            "n_returns_in_year", "share_nil_lines",
                            "last_filing_year", "first_filing_year",
                            "designation", "province", "charity_type",
                            "age_years", "is_section_d"])
print("panel:", panel.shape)
panel.groupby("year").agg(
    n_filers=("bn", "size"),
    pct_section_D=("is_section_d", lambda s: 100 * s.mean()),
    median_revenue=("total_revenue", "median"),
    median_assets=("total_assets", "median"),
).round(1)

### Filing patterns

The panel is remarkably well behaved. Reading each pattern left to right as
2019-2023, `11111` means "filed every year"; `11110` means "last filed in 2022".
Monotone entry and exit dominate; genuine gaps — a charity that skips a year and
returns — are rare but not negligible, and they are what force the censoring rule
in the next section.

In [ ]:
presence = presence_matrix(panel)
pattern = presence.apply(lambda row: "".join(str(v) for v in row), axis=1)
counts = pattern.value_counts()

monotone_exit = {"11110", "11100", "11000", "10000"}
monotone_entry = {"01111", "00111", "00011", "00001"}
gap = [p for p in counts.index if p not in monotone_exit | monotone_entry | {"11111"}
       and p.strip("0").count("0") > 0]

print(f"always present      : {counts.get('11111', 0):>7,}")
print(f"monotone exit       : {sum(counts.get(p, 0) for p in monotone_exit):>7,}")
print(f"monotone entry      : {sum(counts.get(p, 0) for p in monotone_entry):>7,}")
print(f"gap-then-return     : {sum(counts[p] for p in gap):>7,}"
      f"   ({100 * sum(counts[p] for p in gap) / len(pattern):.2f}% of BNs)")
counts.head(12).to_frame("n_charities")

## 4. What counts as failure

Two outcomes, for two different questions.

### `exit` — permanent deregistration (our primary outcome)

A registered charity must file a T3010 every year it remains registered. When a
BN stops appearing, its registration has ended: voluntary revocation, revocation
for cause, revocation for failure to file, amalgamation, or wind-up. That is the
closest thing to corporate failure the data contains, and unlike the
net-asset-decline proxy it is an event, not a threshold.

Two honest caveats, stated here rather than buried:

* **Disappearance is not always failure.** A charity that merges into a larger
  one, or that winds up solvent and by choice, exits the panel identically to one
  that collapses. Every result below predicts *deregistration*, not insolvency.
* **Absence needs confirming.** A charity missing from year *t+1* might simply
  be filing late relative to the extract date. We require at least
  `MIN_LOOKAHEAD_YEARS` observed years after *t* before calling an absence
  permanent, which restricts labelled years to 2019-2021.

### `vulnerable` — the Greenlee-Trussel three-year net-asset decline

Net assets falling 20% or more over three years. Included because it is the
outcome the benchmark models were designed for; judging them only on an outcome
they were not built for would not be a fair test. With a five-year window it can
be resolved for the 2019 and 2020 cohorts only.

In [ ]:
panel = add_outcomes(panel)
summary = outcome_summary(panel)
print("years with a trustworthy exit label:", labelled_years(), "\n")
summary.round(4)

The 2022 row shows why the lookahead rule exists. Its *provisional* exit rate is
2.9%, well above the 1.6-2.0% of the confirmed years — but 268 charities absent
in 2022 reappear in 2023, so roughly one in eight single-year absences is a gap
rather than an exit. Treating 2022 as labelled would train the model partly on
late filing. It is excluded from the labelled analysis and used instead as the
scoring cohort in `03-model-results`.

In [ ]:
fig, ax = plots.plt.subplots(figsize=(7.2, 4.2))
plots.plot_exit_rates(summary, ax=ax);

### Who exits

Exits are concentrated among small, young charities — which is exactly why a
size-only baseline is a demanding benchmark, and why the classics' size control
does so much work.

In [ ]:
labelled = panel.loc[panel["exit_next_year"].notna()].copy()
labelled["revenue_band"] = pd.cut(
    labelled["total_revenue"],
    [-np.inf, 0, 10_000, 100_000, 1_000_000, np.inf],
    labels=["nil or negative", "< $10k", "$10k-$100k", "$100k-$1M", "> $1M"],
)
pd.DataFrame({
    "n_charity_years": labelled.groupby("revenue_band", observed=True).size(),
    "exit_rate": labelled.groupby("revenue_band", observed=True)["exit_next_year"].mean(),
    "share_of_all_exits": (labelled.groupby("revenue_band", observed=True)["exit_next_year"].sum()
                           / labelled["exit_next_year"].sum()),
}).style.format({"n_charity_years": "{:,.0f}", "exit_rate": "{:.2%}",
                 "share_of_all_exits": "{:.1%}"})

In [ ]:
by_age = labelled.assign(
    age_band=pd.cut(labelled["age_years"], [-np.inf, 5, 15, 30, 50, np.inf],
                    labels=["< 5y", "5-15y", "15-30y", "30-50y", "50y+"])
).groupby("age_band", observed=True)["exit_next_year"].agg(["size", "mean"])
by_age.columns = ["n_charity_years", "exit_rate"]
by_age.style.format({"n_charity_years": "{:,.0f}", "exit_rate": "{:.2%}"})

## 5. The temporal design

Features for year *t* use levels at *t* and growth from *t-1*; the exit label
needs at least two later years. In a 2019-2023 window that leaves exactly two
fully-specified years, and the split writes itself:

| | year | features from | label resolved against |
|---|---|---|---|
| **train** | 2020 | 2019-2020 | 2021-2023 |
| **test** | 2021 | 2020-2021 | 2022-2023 |
| **score** | 2022 | 2021-2022 | *(not labelled)* |

Training and test share organisations but not periods. That is deliberate: a
monitoring tool is re-fitted each year and applied to the next, so forecasting
forward is the question that matters, not generalising to unseen organisations.